# 13 · Transactional outbox: close the database-to-Kafka gap

**Level:** advanced · **Time:** 50–70 minutes · **Needs:** running Kafka, notebooks 11–12.

Store an order and its event intent in one database transaction. Publish later, keep a
failed event pending, and recreate the inevitable publish-before-mark-sent duplicate.
Then combine the outbox with the durable consumer receipt from notebook 12.

Start with a fresh kernel. This notebook uses a new database and Kafka topic.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


### Jupyter and synchronous Django code

Jupyter kernels may keep an event loop running even with `%autoawait off`. The next cell
installs `%%django_sync`: it executes the visible Python body on one dedicated synchronous
thread and waits for completion. All Django code cells below use that thread, so ORM
connection ownership and transaction boundaries remain consistent. Variables stay in the
notebook namespace, exceptions remain visible, and Django's async safety stays enabled.
The magic adds no business logic. In a terminal, run the same Python bodies normally.


In [ ]:
%autoawait off
sys.path.insert(0, str(ROOT / "django_demo"))
from notebook_support import install_sync_magic
DJANGO_EXECUTOR = install_sync_magic(get_ipython())
print("Synchronous Django cell executor installed")


In [ ]:
%%django_sync
LESSON = "13"

import os
import django
from django.conf import settings

if settings.configured:
    raise RuntimeError("Restart the kernel before running this notebook again; Django settings are process-wide.")
DATA_DIR = ROOT / ".lab-data"
DATA_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / f"django-{LESSON}-{RUN}.sqlite3"
os.environ["DJANGO_DB_PATH"] = str(DB_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
sys.path.insert(0, str(ROOT / "django_demo"))
django.setup()
from django.core.management import call_command
from django.db import transaction
from django.test import Client, override_settings
from django.utils import timezone
from decimal import Decimal
from unittest.mock import patch
from orders.models import Order, OutboxEvent, InboxEvent, OrderProjection
from orders.services import (
    order_event, validate_event, publish_event, apply_event,
    create_order_after_commit, create_order_with_outbox, publish_pending,
)
call_command("migrate", verbosity=0)
http = Client()
print("Isolated database:", DB_PATH.name)


## Step 1 — Examine the atomic write

`Order` is business state. `OutboxEvent` is the durable intent to publish that change.
Both use the same database transaction and connection. A database commit preserves
both; a rollback preserves neither. The request does not contact Kafka.


In [ ]:
%%django_sync
import inspect
print(inspect.getsource(create_order_with_outbox))
require_broker()
topic = new_topic("django-outbox", partitions=1)


## Step 2 — Accept a request without sending a message

A spy verifies that the HTTP endpoint never invokes the network publisher. Returning
201 means the order and pending event are durable; it does not mean a consumer has
projected the order yet.


In [ ]:
%%django_sync
with override_settings(KAFKA_ORDERS_TOPIC=topic), patch("orders.services.publish_event") as send:
    response = http.post("/orders/", {"customer": "Outbox learner", "total": "25.00"}, content_type="application/json")
    send.assert_not_called()
assert response.status_code == 201, response.content
result = response.json()
pending = OutboxEvent.objects.get(event_id=result["event_id"])
assert pending.published_at is None
assert http.get(f'/projections/{result["order_id"]}/').status_code == 404
print(result)


## Step 3 — Prove atomic rollback

Introduce an error in an outer transaction after creating an order and outbox row.
Neither should survive. The test uses actual database writes and a caught exception.


In [ ]:
%%django_sync
before = (Order.objects.count(), OutboxEvent.objects.count())
try:
    with transaction.atomic():
        create_order_with_outbox("Will roll back", "2.00", topic=topic)
        raise RuntimeError("request failed before commit")
except RuntimeError:
    pass
assert before == (Order.objects.count(), OutboxEvent.objects.count())
print("Order and event intent rolled back together")


## Step 4 — Inspect the publisher's ordering

The publisher reads pending rows, sends one event, waits for its delivery acknowledgement,
and only then sets `published_at`. It runs a finite snapshot. The time budget is checked
between rows; an individual send can take up to 15 seconds beyond that budget.


In [ ]:
%%django_sync
print(inspect.getsource(publish_pending))


## Step 5 — Failure must leave the event pending

Inject a delivery error. A publisher must not mark an event as published merely because
`produce()` accepted it into a local buffer. The callback/flush result establishes delivery.


In [ ]:
%%django_sync
with patch("orders.services.publish_event", side_effect=RuntimeError("simulated delivery failure")):
    try:
        publish_pending(topic=topic)
    except RuntimeError as exc:
        print(exc)
pending.refresh_from_db()
assert pending.published_at is None
print("Pending event is available for retry")


## Step 6 — Retry against the real broker

The management command uses the same service. It exits when the bounded snapshot is
drained; schedule or supervise it separately in a real system. For this SQLite lesson,
run exactly one outbox publisher at a time.


In [ ]:
%%django_sync
call_command("publish_outbox", topic=topic, max_messages=10, timeout=20)
pending.refresh_from_db()
assert pending.published_at is not None
first_message = collect(topic, count=1)[0]
assert decode(first_message)["event_id"] == result["event_id"]
print("Published at:", pending.published_at)


## Step 7 — Marked rows are skipped

Calling the publisher again must not send another event for already marked rows.
This assertion inspects the returned count and verifies that the network function
was not called; it does not rely on a short Kafka polling timeout to prove absence.


In [ ]:
%%django_sync
with patch("orders.services.publish_event") as send:
    assert publish_pending(topic=topic) == 0
    send.assert_not_called()
print("No pending rows; no publish attempted")


## Step 8 — Recreate a crash AFTER delivery, BEFORE marking

Manually publish a second pending event, then deliberately leave its row unmarked.
This models a process dying immediately after a successful acknowledgement. The
replacement publisher cannot know delivery succeeded, so it correctly retries.


In [ ]:
%%django_sync
second_order, second_event = create_order_with_outbox("Crash window", "31.50", topic=topic)
publish_event(second_event, topic)
second_row = OutboxEvent.objects.get(event_id=second_event["event_id"])
assert second_row.published_at is None
assert publish_pending(topic=topic) == 1
print("One business event was delivered twice")


## Step 9 — Observe three records but two distinct event IDs

A fresh group reads the whole topic from the beginning. The repeated event keeps its
original ID and payload. Creating a new event ID on retry would defeat deduplication.


In [ ]:
%%django_sync
records = collect(topic, count=3)
event_ids = [decode(record)["event_id"] for record in records]
assert len(event_ids) == 3
assert len(set(event_ids)) == 2
assert event_ids.count(second_event["event_id"]) == 2
print(event_ids)


## Step 10 — Apply all deliveries safely

The outbox is an at-least-once publication mechanism. Combine it with durable consumer
receipts to make database effects idempotent. The inbox and projection share one database
transaction, while Kafka commits remain a separate operation.


In [ ]:
%%django_sync
effects = [apply_event(decode(record)) for record in records]
assert effects.count(True) == 2 and effects.count(False) == 1
assert OrderProjection.objects.count() == InboxEvent.objects.count() == 2
print("Applied flags:", effects)


## Scaling discussion and exercises

1. Add `attempt_count` and `last_error` fields. Keep failures pending and expose their age.
2. Why is setting `published_at` before sending unsafe? Explain the lost-message window.
3. Why doesn't a Kafka transaction make SQLite and Kafka one atomic transaction?
4. Propose a PostgreSQL design for multiple publishers, preserving per-order ordering.

**Solution hints:** acknowledgement-before-marking favors duplicates over silent loss;
Kafka transactions do not enlist this SQLite database. Multiple publishers need an explicit
claim/lease or locking protocol and partition/order policy. SQLite's `select_for_update()`
does not provide the row-locking design you would need. This sample deliberately supports
one publisher. A production system may use CDC for the outbox; operating a connector,
schema evolution, retention, monitoring, and recovery are additional responsibilities.

Pending rows and receipts need retention policies. Never purge receipts while events that
rely on them can still be replayed. HTTP request idempotency is separate from event deduplication.
